In [ ]:
# pip install presidio-analyzer presidio-anonymizer spacy azureml-fsspec pandas gliner2
# python -m spacy download en_core_web_lg

from pathlib import Path
import io
import time
import pandas as pd
from azureml.fsspec import AzureMachineLearningFileSystem
from redaction import PIIRedactor

In [ ]:
INPUT_URI = "azureml://subscriptions/<sub>/resourcegroups/<rg>/workspaces/<ws>/datastores/share_datastore/paths/yvonne-datastore/raw-call-transcripts/"

OUT_DIR = "<fill in output path here>"

SCORE_THRESHOLD = 0.5
MAX_FILES = 1   # set to None to run all files
TARGET_FILE = None  # e.g. "example_1.csv" to run one specific file, else None

In [ ]:
pii_redactor = PIIRedactor()

# Entities to redact.
# PIIRedactor.PII_ENTITIES covers the core SG-specific set.
# ACCOUNT_NUMBER is registered in PIIRedactor but excluded from the default list — added explicitly here.
ENTITIES = PIIRedactor.PII_ENTITIES + [
    "ACCOUNT_NUMBER",
]

In [ ]:
# debug=True prints for every row:
#   - the raw text of the row
#   - result: the redacted text returned by PIIRedactor
#
# To debug a specific file:
#   df = redact_csv(data, debug=True)
#
# To debug a small set of rows without loading a file:
#   rows = ["the account number is 9323284844", "just give me a moment", "ok", "9323284844"]
#   redact_rows(rows, debug=True)
def redact_rows(rows, debug=False):
    redacted = []
    for i, text in enumerate(rows):
        cleaned_text = pii_redactor.redact(text=text, entities=ENTITIES, score_threshold=SCORE_THRESHOLD, use_tags=True)
        if debug:
            print(f"\n--- row {i} ---")
            print(f"text:    {text!r}")
            print(f"result:  {cleaned_text!r}")
        redacted.append(cleaned_text)
    return redacted


def redact_csv(data, debug=False):
    df = pd.read_csv(io.BytesIO(data))
    if "text" not in df.columns:
        raise ValueError("No 'text' column found in CSV")
    rows = df["text"].fillna("").tolist()
    df["text"] = redact_rows(rows, debug=debug)
    return df

In [ ]:
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

fs = AzureMachineLearningFileSystem(INPUT_URI)
folder = INPUT_URI.split("/paths/", 1)[1].rstrip("/")

if TARGET_FILE:
    csv_files = [TARGET_FILE]
else:
    csv_files = sorted(f for f in fs.ls("") if f.endswith(".csv"))
    if MAX_FILES:
        csv_files = csv_files[:MAX_FILES]

print(f"Found {len(csv_files)} CSV file(s)\n")

total_start = time.time()

for i, filepath in enumerate(csv_files, 1):
    filename = Path(filepath).name
    print(f"[{i}/{len(csv_files)}] {filename} ...", end=" ", flush=True)

    file_start = time.time()

    with fs.open(f"{folder}/{filename}", "rb") as f:
        data = f.read()

    df = redact_csv(data)
    df.to_csv(out_dir / filename, index=False)

    elapsed = time.time() - file_start
    print(f"done ({len(df)} rows, {elapsed:.1f}s, {elapsed/len(df):.2f}s/row)")

total_elapsed = time.time() - total_start
print(f"\nAll done in {total_elapsed:.1f}s. Redacted CSVs saved to: {out_dir}")